# Introduction

We have a dataset about collection of processed and compiled records of experimental heat flux and boundary conditions used for the work presented in article. There are nine variables contained in the dataset:

* Author
* Geometry
* Pressure
* Mass Flux
* x_e_out
* D_e
* D_h
* Length
* chf_exp

Our task in this competition is to impute all missing values and submit them. There is no test dataset in this competition.

# Loading Libraries

In [ ]:
!pip install pygam==0.8.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

from category_encoders import OneHotEncoder, MEstimateEncoder, GLMMEncoder, OrdinalEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, KFold
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor, StackingRegressor, AdaBoostRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor, LogisticRegression
from sklearn.linear_model import PassiveAggressiveRegressor, ARDRegression
from sklearn.linear_model import TheilSenRegressor, RANSACRegressor, HuberRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score, roc_curve
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from pygam import LinearGAM
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn import set_config

set_config(transform_output = 'pandas')

sns.set_theme(style = 'white', palette = 'viridis')
pal = sns.color_palette('viridis')

pd.set_option('display.max_rows', 100)

In [ ]:
data = pd.read_csv(r'../input/playground-series-s3e15/data.csv')
orig_data = pd.read_csv(r'../input/predicting-heat-flux/Data_CHF_Zhao_2020_ATE.csv')

data.drop('id', axis = 1, inplace = True)
orig_data.drop('id', axis = 1, inplace = True)

# Dataset Diagnostics

In [ ]:
data.head(10)

In [ ]:
desc = pd.DataFrame(index = list(data))
desc['count'] = data.count()
desc['nunique'] = data.nunique()
desc['%unique'] = desc['nunique'] / len(data) * 100
desc['null'] = data.isnull().sum()
desc['type'] = data.dtypes
desc = pd.concat([desc, data.describe().T], axis = 1)
desc

In [ ]:
desc = pd.DataFrame(index = list(orig_data))
desc['count'] = orig_data.count()
desc['nunique'] = orig_data.nunique()
desc['%unique'] = desc['nunique'] / len(orig_data) * 100
desc['null'] = orig_data.isnull().sum()
desc['type'] = orig_data.dtypes
desc = pd.concat([desc, orig_data.describe().T], axis = 1)
desc

**Key points**: 
1. We have two categorical features and seven numerical features. We can group them to make our code writing easier.
2. We have TONS of missing values in the competition dataset. Our task is to impute them.
3. `chf_exp [MW/m2]` is the only feature without missing value.
4. `D_e [mm]`, `D_h [mm]`, and `length [mm]` can be considered as categorical due to the low amount of unique values. We might also consider `pressure [MPa]` and `chf_exp [MW/m2]` as one too if we want to go to the extreme. However, for simplicity, I will not consider them as one for this notebook.

In [ ]:
categorical_features = data.columns[:2]
numerical_features = data.columns[2:]

# Distribution of Numerical Features

In [ ]:
fig, ax = plt.subplots(7, 1, figsize = (7, 20), dpi = 300)
ax = ax.flatten()

for i, column in enumerate(numerical_features):
    sns.kdeplot(data[column], ax=ax[i], color=pal[0])
    sns.kdeplot(orig_data[column], ax=ax[i], color=pal[2])
    
    ax[i].set_title(f'{column} Distribution', size = 7)
    ax[i].set_xlabel(None)
    ax[i].set_ylabel(None)
    
fig.suptitle('Distribution of Feature\n\n', fontsize = 15, fontweight = 'bold')
#fig.legend(['Train', 'Original Train'])
plt.tight_layout()

**Key point:** There are a lot of differences between original dataset distribution and competition dataset distribution, which is understandable considering the amount of missing data we have.

# Distribution of Categorical Features

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    data['author'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 10)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = data, y = 'author', ax = ax[1], palette = 'viridis', order = data['author'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Author in Competition Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    orig_data['author'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 10)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = orig_data, y = 'author', ax = ax[1], palette = 'viridis', order = orig_data['author'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Author in Original Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    data['geometry'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 3)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = data, y = 'geometry', ax = ax[1], palette = 'viridis')
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Geometry in Competition Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    orig_data['geometry'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(0, 3)], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = orig_data, y = 'geometry', ax = ax[1], palette = 'viridis')
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Geometry in Original Dataset', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

**Key point**: Both `author` and `geometry` have imbalanced distribution.

# Correlation

In [ ]:
def heatmap(dataset, label = None):
    corr = dataset.corr(method = 'spearman')
    plt.figure(figsize = (10, 10), dpi = 300)
    mask = np.zeros_like(corr)
    mask[np.triu_indices_from(mask)] = True
    sns.heatmap(corr, mask = mask, cmap = 'viridis', annot = True, annot_kws = {'size' : 12})
    plt.title(f'{label} Dataset Correlation Matrix\n', fontsize = 25, weight = 'bold')
    plt.show()

In [ ]:
heatmap(data[numerical_features], 'Competition')
heatmap(orig_data[numerical_features], 'Original')

**Key point**: Most of the numerical features are correlated, especially between `D_e [mm]` and `D_h [mm]`.

In [ ]:
def distance(data, label = ''):
    #thanks to @sergiosaharovsky for the fix
    corr = data.corr(method = 'spearman')
    dist_linkage = linkage(squareform(1 - abs(corr)), 'complete')
    
    plt.figure(figsize = (10, 8), dpi = 300)
    dendro = dendrogram(dist_linkage, labels=data.columns, leaf_rotation=90)
    plt.title(f'Feature Distance in {label} Dataset', weight = 'bold', size = 22)
    plt.show()

In [ ]:
distance(data[numerical_features], 'Competition')
distance(orig_data[numerical_features], 'Original')

**Key point**: Both original dataset and competition dataset have similar feature clustering.

# Categorical Feature Preprocessing

Our categorical feature pre-process will be as follows:
1. Impute categorical features by filling them with the most common category.
2. Do one-hot encoding on categorical features.

In [ ]:
class SimpleCategoricalImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, x, y = None):
        return self
    def transform(self, x, y = None):
        x_copy = x.copy()
        x_copy['author'] = x_copy['author'].fillna('Thompson')
        x_copy['geometry'] = x_copy['geometry'].fillna('tube')
        return x_copy

In [ ]:
CategoricalPreprocessor = Pipeline([
    ('categorical_imputer', SimpleCategoricalImputer()),
    ('onehot', OneHotEncoder(cols = categorical_features)),
])

In [ ]:
preprocessed_data = CategoricalPreprocessor.fit(data).transform(data)
preprocessed_orig_data = CategoricalPreprocessor.transform(orig_data)

Before going on, we need to do preprocessing on our column names too. This is because XGBoost can't accept column name with `[`, `]`, and `<` character.

In [ ]:
#https://stackoverflow.com/questions/48645846/pythons-xgoost-valueerrorfeature-names-may-not-contain-or
regex = regex = re.compile(r"\[|\]|<", re.IGNORECASE)
preprocessed_data.columns = [regex.sub("_", col) if any(x in str(col) for x in set(('[', ']', '<'))) else col for col in preprocessed_data.columns.values]
preprocessed_orig_data.columns = [regex.sub("_", col) if any(x in str(col) for x in set(('[', ']', '<'))) else col for col in preprocessed_orig_data.columns.values]

All brackets have been replaced with underscore(`_`) now.

# Regression Preparation

I will split our data based on whether `x_e_out _-_` is null or not. If it's null, it'll be included into test dataset. If it's not, it'll be included into train dataset. The reason why I choose that feature is that, it's the only feature that should be submitted in this competition, therefore we can safely assume that we can treat this as regression problem with `x_e_out _-_` as the target.

In [ ]:
train = preprocessed_data[preprocessed_data['x_e_out _-_'].isna() == False]
test = preprocessed_data[preprocessed_data['x_e_out _-_'].isna() == True]
test = test.drop('x_e_out _-_', axis = 1)

In [ ]:
X = train.copy()
y = X.pop('x_e_out _-_')

seed = 42
splits = 5
np.random.seed(seed)

k = KFold(n_splits = 5, random_state = seed, shuffle = True)

PyGAM is incompatible with scikit-learn ensemblers. Therefore, we have to create our own wrapper so it can be included in `VotingRegressor`.

In [ ]:
def gam_wrapper(gam_model):
    class GAMWrapper(BaseEstimator, RegressorMixin):
        def fit(self, X, y):
            self.estimator_ = gam_model
            self.estimator_.fit(X, y)
            return self
        def predict(self, X):
            y = self.estimator_.predict(X)
            return y
    return GAMWrapper()

# Adversarial Validation

Before we start doing cross-validation, we should see if our train dataset is similar to test dataset. One good way to do this is adversarial validation. Basically, we want to measure the similarity of datasets so we can be sure if we can generalize the model from train dataset into test dataset. If the validation results in ROC-AUC score of close to .5, both train and test dataset are indistinguishable, therefore we can trust our CV.

In [ ]:
def adversarial_validation(dataset_1 = train, dataset_2 = test, label = 'Train-Test'):
    
    #thanks to @carlmcbrideellis
    #https://www.kaggle.com/code/carlmcbrideellis/what-is-adversarial-validation

    adv_train = dataset_1.drop('x_e_out _-_', axis = 1)
    adv_test = dataset_2.copy()

    adv_train['is_test'] = 0
    adv_test['is_test'] = 1

    adv = pd.concat([adv_train, adv_test], ignore_index = True)

    adv_shuffled = adv.sample(frac = 1)

    adv_X = adv_shuffled.drop('is_test', axis = 1)
    adv_y = adv_shuffled.is_test

    skf = StratifiedKFold(n_splits = 5, random_state = 42, shuffle = True)

    val_scores = []
    predictions = np.zeros(len(adv))

    for fold, (train_idx, val_idx) in enumerate(skf.split(adv_X, adv_y)):
    
        adv_lr = XGBClassifier(random_state = 42)    
        adv_lr.fit(adv_X.iloc[train_idx], adv_y.iloc[train_idx])
        
        val_preds = adv_lr.predict_proba(adv_X.iloc[val_idx])[:,1]
        predictions[val_idx] = val_preds
        val_score = roc_auc_score(adv_y.iloc[val_idx], val_preds)
        val_scores.append(val_score)
    
    fpr, tpr, _ = roc_curve(adv['is_test'], predictions)
    
    plt.figure(figsize = (10, 10), dpi = 300)
    sns.lineplot(x=[0, 1], y=[0, 1], linestyle="--", label="Indistinguishable Datasets")
    sns.lineplot(x=fpr, y=tpr, label="Adversarial Validation Classifier")
    plt.title(f'{label} Validation = {np.mean(val_scores):.5f}', weight = 'bold', size = 17)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.show()

In [ ]:
adversarial_validation()

**Key point**: Train and test dataset are similar because the score is very close to .5, therefore we can trust our CV.

# Cross-Validation Function

In [ ]:
def cross_val_score(model, cv = k, label = '', include_original = False):
    
    X = train.copy()
    y = X.pop('x_e_out _-_')
    
    #initiate prediction arrays and score lists
    val_predictions = np.zeros((len(train)))
    predictions = np.zeros((len(test)))
    #train_predictions = np.zeros((len(train)))
    train_scores, val_scores = [], []
    
    #training model, predicting prognosis probability, and evaluating log loss
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        
        #define train set
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        
        #concat train set with original dataset if include_original is True
        if include_original == True:
            X_train = pd.concat([X_train, preprocessed_orig_data.drop('x_e_out _-_', axis = 1)])
            y_train = pd.concat([y_train, preprocessed_orig_data['x_e_out _-_']])
        
        #define validation set
        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]
        
        #train model
        model.fit(X_train, y_train)
        
        #make predictions
        train_preds = model.predict(X_train)
        val_preds = model.predict(X_val)
        
        predictions += model.predict(test) / cv.get_n_splits()
                  
        val_predictions[val_idx] += val_preds
        
        #evaluate model for a fold
        train_score = mean_squared_error(y_train, train_preds, squared = False)
        val_score = mean_squared_error(y_val, val_preds, squared = False)
        
        #append model score for a fold to list
        train_scores.append(train_score)
        val_scores.append(val_score)
    
    print(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | Train Score: {np.mean(train_scores):.5f} ± {np.std(train_scores):.5f} | {label}')
    
    return val_scores, val_predictions, predictions

# Baseline Models I

In order to get reliable CV, I will do imputation inside the CV process.

In [ ]:
score_list, oof_list, predict_list = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

models = [
    ('linear', LinearRegression()),
    ('ridge', Ridge(random_state = seed)),
    ('lasso', Lasso(random_state = seed, max_iter = 1000000)),
    ('elastic', ElasticNet(random_state = seed, max_iter = 1000000)),
    ('huber', HuberRegressor(max_iter = 1000000)),
    ('ard', ARDRegression()),
    ('passive', PassiveAggressiveRegressor(random_state = seed)),
    ('theilsen', TheilSenRegressor(random_state = seed)),
    ('gam', gam_wrapper(LinearGAM())),
    #('mlp', MLPRegressor(random_state = seed, max_iter = 1000000)),
    ('et', ExtraTreesRegressor(random_state = seed)),
    ('rf', RandomForestRegressor(random_state = seed)),
    ('xgb', XGBRegressor(random_state = seed)),
    ('lgb', LGBMRegressor(random_state = seed)),
    ('dart', LGBMRegressor(random_state = seed, boosting_type = 'dart')),
    ('cb', CatBoostRegressor(random_state = seed, verbose = 0)),
    ('gb', GradientBoostingRegressor(random_state = seed)),
    ('hgb', HistGradientBoostingRegressor(random_state = seed)),
    ('ada', AdaBoostRegressor(random_state = seed)),
    ('knn', KNeighborsRegressor())
]

for (label, model) in models:
     score_list[label], oof_list[label], predict_list[label] = cross_val_score(
         Pipeline([
             ('impute', SimpleImputer()),
             (label, model)
         ]),
         label = label)

In [ ]:
plt.figure(figsize = (8, 4), dpi = 300)
sns.barplot(data = score_list.reindex((score_list).mean().sort_values().index, axis = 1), palette = 'viridis', orient = 'h')
plt.title('Score Comparison', weight = 'bold', size = 20)
plt.show()

**Key points**: 
1. Linear models can perform as well as tree-based models. However, gradient boosting models outperform everything.
2. Our best performing models is LightGBM, followed by CatBoost and Histogram Gradient Boosting Regressor.
3. Surprisingly most of our tree-based models aren't overfitted, with exception of Extra Trees and maybe Random Forest.

**Notes**: There are two secions that I've deleted to simplify this notebook. Those are baseline models with original data and scaling. In short, using original data doesn't help this notebook models, and while scaling helps most of the linear models, it doesn't help tree-based models and GAM.

# Ensemble

For simplicity, we'll use Ridge regression to define the weight for our voting ensemble.

In [ ]:
weights = Ridge(random_state = seed, positive = False, fit_intercept = False).fit(X = oof_list, y = y).coef_

pd.DataFrame(weights, index = oof_list.columns, columns = ["Weight per Model"])

In [ ]:
voter = Pipeline([
    ('impute', SimpleImputer()),
    ('vote', VotingRegressor(models, weights = weights))
])

_, _, prediction = cross_val_score(voter, label = 'Voting Ensemble')

# Submission

In [ ]:
submission = pd.DataFrame(test.index, columns = ['id'])
submission['x_e_out [-]'] = prediction
submission.to_csv('submission.csv', index = False)

Thanks for reading!